# 🚀 DataCo Supply Chain — BigQuery Deployment
**CO4033 — ĐH Bách Khoa TP.HCM | Phụ trách: Quốc Khánh**

## Cách chạy:
1. Upload file `btl_deploy.zip` vào Colab (panel 📁 bên trái → ⬆️)
2. Sửa **Project ID** ở Cell 2
3. **Runtime → Run all** → chờ ~5 phút ✅

In [ ]:
# ============================================================
# CELL 1 — Cài thư viện + Giải nén ZIP
# ============================================================
import zipfile, os
from pathlib import Path

# Cài thư viện
import subprocess
subprocess.run(['pip', 'install', 'google-cloud-bigquery', 'pyarrow', 'pandas', 'db-dtypes', '-q'])
print('✅ Cài xong thư viện')

# Giải nén btl_deploy.zip
zip_path = Path('/content/btl_deploy.zip')
if zip_path.exists():
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/')
    print(f'✅ Giải nén xong: {zip_path.name}')
else:
    print('❌ Không tìm thấy btl_deploy.zip!')
    print('   → Hãy upload file btl_deploy.zip vào Colab trước (panel 📁 bên trái → ⬆️)')
    raise FileNotFoundError('Thiếu btl_deploy.zip')

# Xác nhận folder gold/
gold_dir = Path('/content/gold')
if gold_dir.exists():
    files = list(gold_dir.glob('*.parquet'))
    print(f'✅ Thư mục gold/ có {len(files)} Parquet files:')
    for f in sorted(files):
        print(f'   - {f.name} ({f.stat().st_size/1024/1024:.2f} MB)')
else:
    raise FileNotFoundError('❌ Không tìm thấy /content/gold/ sau khi giải nén!')

In [ ]:
# ============================================================
# CELL 2 — ⚠️ ĐIỀN PROJECT ID CỦA BẠN VÀO ĐÂY
# Lấy tại: console.cloud.google.com (góc trên bên trái)
# ============================================================

PROJECT_ID = 'THAY_BANG_PROJECT_ID_CUA_BAN'  # ← SỬA DÒNG NÀY
DATASET_ID = 'dataco_logistics_gold'

# ---- Không sửa bên dưới ----
if PROJECT_ID == 'THAY_BANG_PROJECT_ID_CUA_BAN':
    raise ValueError('❌ Bạn chưa điền Project ID! Sửa dòng PROJECT_ID phía trên.')

os.environ['GCP_PROJECT_ID'] = PROJECT_ID
GOLD_DIR = Path('/content/gold')
print(f'✅ Project ID : {PROJECT_ID}')
print(f'✅ Dataset    : {DATASET_ID}')
print(f'✅ Gold dir   : {GOLD_DIR}')

In [ ]:
# ============================================================
# CELL 3 — Xác thực Google Account
# (Popup sẽ xuất hiện → chọn Gmail của bạn → cho phép)
# ============================================================
from google.colab import auth
auth.authenticate_user()
print('✅ Đã xác thực Google Account')

In [ ]:
# ============================================================
# CELL 4 — ⭐ LOAD 6 BẢNG LÊN BIGQUERY
# Mất ~2–5 phút tùy tốc độ mạng
# ============================================================
import pandas as pd
from google.cloud import bigquery
from pathlib import Path

GOLD_DIR = Path('/content/gold')
client   = bigquery.Client(project=PROJECT_ID)

# Tạo Dataset
dataset_ref = bigquery.Dataset(f'{PROJECT_ID}.{DATASET_ID}')
dataset_ref.location = 'US'
client.create_dataset(dataset_ref, exists_ok=True)
print(f'✅ Dataset [{DATASET_ID}] sẵn sàng\n')

# Config mặc định
base_config = bigquery.LoadJobConfig(
    write_disposition='WRITE_TRUNCATE',
    autodetect=True,
)

# Config riêng cho fact table (Partition + Cluster)
fact_config = bigquery.LoadJobConfig(
    write_disposition='WRITE_TRUNCATE',
    autodetect=True,
    time_partitioning=bigquery.TimePartitioning(
        type_=bigquery.TimePartitioningType.MONTH,
        field='order_date_ts',
    ),
    clustering_fields=['shipping_ops_key', 'destination_geo_key', 'late_delivery_risk'],
)

TABLES = [
    ('fact_order_shipments', fact_config),
    ('dim_date',             base_config),
    ('dim_customer',         base_config),
    ('dim_product',          base_config),
    ('dim_geography',        base_config),
    ('dim_shipping_ops',     base_config),
]

print('🚀 Bắt đầu load lên BigQuery...')
for table_name, config in TABLES:
    parquet_path = GOLD_DIR / f'{table_name}.parquet'
    if not parquet_path.exists():
        print(f'  ❌ Không tìm thấy: {parquet_path}')
        continue
    df        = pd.read_parquet(parquet_path)
    table_ref = f'{PROJECT_ID}.{DATASET_ID}.{table_name}'
    print(f'  ⬆️  {table_name} ({len(df):,} dòng)...', end=' ', flush=True)
    job = client.load_table_from_dataframe(df, table_ref, job_config=config)
    job.result()
    print('✅')

print('\n🎉 Load xong tất cả 6 bảng!')

In [ ]:
# ============================================================
# CELL 5 — ⭐ TẠO 4 SEMANTIC VIEWS
# ============================================================
from google.cloud import bigquery
P      = PROJECT_ID
D      = DATASET_ID
client = bigquery.Client(project=P)

VIEWS = {
    # View 1 — Quốc Khánh: Executive KPI & Global Map
    'vw_exec_overview': f"""
        SELECT f.order_item_id, f.order_id, f.order_date_ts,
            dd.year AS order_year, dd.month AS order_month,
            dd.quarter AS order_quarter, dd.is_peak_season,
            dg.market, dg.order_region, dg.order_country, dg.order_city,
            ds.shipping_mode, ds.delivery_status, ds.order_status,
            f.net_sales, f.net_profit, f.discount_rate, f.order_item_quantity,
            f.late_delivery_risk, f.delay_days, f.is_loss_order,
            f.is_canceled_or_fraud, f.days_shipment_scheduled, f.days_shipping_real
        FROM `{P}.{D}.fact_order_shipments` f
        LEFT JOIN `{P}.{D}.dim_date`         dd ON f.order_date_key      = dd.date_key
        LEFT JOIN `{P}.{D}.dim_geography`    dg ON f.destination_geo_key = dg.destination_geo_key
        LEFT JOIN `{P}.{D}.dim_shipping_ops` ds ON f.shipping_ops_key    = ds.shipping_ops_key
    """,
    # View 2 — Vinh Khánh: OLAP Diagnostic
    'vw_olap_diagnostic': f"""
        SELECT f.order_item_id, f.order_id, f.order_date_ts, f.shipping_date_ts,
            dd.year, dd.month, dd.quarter, dd.day_of_week, dd.is_weekend, dd.is_peak_season,
            dc.customer_segment,
            dg.market, dg.order_region, dg.order_country, dg.order_city,
            dp.department_name, dp.category_name, dp.product_name,
            ds.shipping_mode, ds.payment_type, ds.order_status,
            ds.delivery_status, ds.default_sla_days,
            f.days_shipment_scheduled, f.days_shipping_real, f.delay_days,
            f.late_delivery_risk, f.net_sales, f.net_profit,
            f.discount_rate, f.order_item_quantity, f.is_loss_order
        FROM `{P}.{D}.fact_order_shipments` f
        LEFT JOIN `{P}.{D}.dim_date`         dd ON f.order_date_key      = dd.date_key
        LEFT JOIN `{P}.{D}.dim_customer`     dc ON f.customer_key        = dc.customer_key
        LEFT JOIN `{P}.{D}.dim_geography`    dg ON f.destination_geo_key = dg.destination_geo_key
        LEFT JOIN `{P}.{D}.dim_product`      dp ON f.product_key         = dp.product_key
        LEFT JOIN `{P}.{D}.dim_shipping_ops` ds ON f.shipping_ops_key    = ds.shipping_ops_key
    """,
    # View 3 — Quân: ML Pre-shipment (KHÔNG có target leakage columns)
    'vw_ml_pre_shipment': f"""
        SELECT f.order_item_id,
            ds.shipping_mode, ds.payment_type, ds.default_sla_days,
            f.days_shipment_scheduled,
            dg.market, dg.order_region, dg.order_country,
            dc.customer_segment,
            dp.department_name, dp.category_name, dp.catalog_price,
            f.order_item_quantity, f.discount_rate, f.net_sales,
            EXTRACT(HOUR FROM f.order_date_ts) AS order_hour, dd.day_of_week AS order_dayofweek,
            dd.is_weekend, dd.is_peak_season,
            dd.month AS order_month, dd.quarter AS order_quarter,
            f.late_delivery_risk
        FROM `{P}.{D}.fact_order_shipments` f
        LEFT JOIN `{P}.{D}.dim_date`         dd ON f.order_date_key      = dd.date_key
        LEFT JOIN `{P}.{D}.dim_customer`     dc ON f.customer_key        = dc.customer_key
        LEFT JOIN `{P}.{D}.dim_geography`    dg ON f.destination_geo_key = dg.destination_geo_key
        LEFT JOIN `{P}.{D}.dim_product`      dp ON f.product_key         = dp.product_key
        LEFT JOIN `{P}.{D}.dim_shipping_ops` ds ON f.shipping_ops_key    = ds.shipping_ops_key
    """,
    # View 4 — Vương: Strategy & Profit Optimization
    'vw_strategy_optimization': f"""
        SELECT f.order_item_id, f.order_id, f.order_date_ts,
            dg.market, dg.order_region, dg.order_country,
            dp.department_name, dp.category_name, dp.product_name,
            ds.shipping_mode, ds.order_status, ds.delivery_status,
            ds.payment_type, ds.default_sla_days,
            f.net_sales, f.net_profit, f.discount_rate, f.order_item_quantity,
            f.delay_days, f.days_shipment_scheduled, f.days_shipping_real,
            f.late_delivery_risk, f.is_loss_order, f.is_canceled_or_fraud,
            dd.year, dd.month, dd.quarter, dd.is_peak_season
        FROM `{P}.{D}.fact_order_shipments` f
        LEFT JOIN `{P}.{D}.dim_date`         dd ON f.order_date_key      = dd.date_key
        LEFT JOIN `{P}.{D}.dim_geography`    dg ON f.destination_geo_key = dg.destination_geo_key
        LEFT JOIN `{P}.{D}.dim_product`      dp ON f.product_key         = dp.product_key
        LEFT JOIN `{P}.{D}.dim_shipping_ops` ds ON f.shipping_ops_key    = ds.shipping_ops_key
    """,
}

print('🔧 Đang tạo 4 Semantic Views...\n')
for view_name, view_sql in VIEWS.items():
    view_ref = f'{P}.{D}.{view_name}'
    view_obj = bigquery.Table(view_ref)
    view_obj.view_query = view_sql
    try:
        client.delete_table(view_ref, not_found_ok=True)
        client.create_table(view_obj)
        print(f'  ✅ {view_name}')
    except Exception as e:
        print(f'  ❌ {view_name}: {e}')

print('\n🎉 Tạo xong 4 Views!')

In [ ]:
# ============================================================
# CELL 6 — Xác nhận & In thông tin gửi cho 3 bạn
# ============================================================
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

print('=' * 60)
print(f'📊 Nội dung dataset [{DATASET_ID}]:')
print('=' * 60)
tables = sorted(
    client.list_tables(f'{PROJECT_ID}.{DATASET_ID}'),
    key=lambda t: t.table_id
)
for t in tables:
    icon = '📋 VIEW ' if t.table_type == 'VIEW' else '📦 TABLE'
    print(f'  {icon}  {t.table_id}')
print('=' * 60)
print(f'✅ Tổng: {len(tables)} bảng/views\n')

print(f'🔗 BigQuery Console:')
print(f'   https://console.cloud.google.com/bigquery?project={PROJECT_ID}\n')

print('━' * 60)
print('📌 THÔNG TIN GỬI CHO 3 BẠN CÒN LẠI')
print('━' * 60)
print('Power BI → Get Data → Google BigQuery → Connect')
print(f'  Project ID : {PROJECT_ID}')
print(f'  Dataset    : {DATASET_ID}')
print()
print('  Vinh Khánh (Trang 2) → vw_olap_diagnostic')
print('  Quân       (Trang 3) → vw_ml_pre_shipment')
print('  Vương      (Trang 4) → vw_strategy_optimization')
print('━' * 60)
print()
print('⚠️  Nhớ share quyền BigQuery cho 3 bạn:')
print('   BigQuery Console → dataco_logistics_gold')
print('   → Sharing → Permissions → Add Principal')
print('   → Role: BigQuery Data Viewer + BigQuery Job User')